In [ ]:
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np

from mdx2.io import loadobj
from mdx2.report.visualization import extract_central_slice, hkl_to_cartesian, unique_slices




# Visualization

Plot slices through the merged reciprocal space map

## Parameters

In [ ]:
# These metadata fields are injected by mdx2.report, regardless of the template. Don't change this line.
_metadata: dict = {}

# Notebook parameters
merged_file: str = None # path to the Nexus file containing the merged hkl_table.
geometry_file: str = None # path to the Nexus file containing symmetry and crystal objects.

In [ ]:
# pretty-print the metadata and parameters
pprint({
    "metadata": _metadata,
    "parameters": {"merged_file": merged_file, "geometry_file": geometry_file}
})

## Results

In [ ]:
hkl_table = loadobj(merged_file, "hkl_table")
symmetry = loadobj(geometry_file,'symmetry')
crystal  = loadobj(geometry_file,'crystal')

### Reciprocal space slices

In [ ]:
# helper function for setting plot limits
def calc_smax(da):
    """Compute maximum s value in cartesian-space xr.DataArray slice"""
    s = np.sqrt(da.sx**2 + da.sy**2)
    smax = s.where(da.notnull()).max().data
    return float(smax)

for slice_index, coords in unique_slices(symmetry).items():
    for c in coords:
        fig, ax = plt.subplots(1,2, figsize=(10,4))
        intensity = extract_central_slice(
            hkl_table,
            symmetry,
            crystal,
            slice_index=slice_index,
            slice_coordinate=0.5*c, # for some grids, this may round to 1/3, etc.
            signal="intensity",
        )
        #uncomment below to also plot the slice with miller index axes
        #intensity.plot(robust=True, vmin=0, ax=ax[1])
        axis_name = intensity.dims[slice_index]
        axis_value = intensity.coords[axis_name].data[0]
        intensity_cartesian = hkl_to_cartesian(intensity, crystal, slice_index=slice_index)
        intensity_cartesian.plot.pcolormesh("sx", "sy", vmin=0, robust=True, ax=ax[0])
        smax = calc_smax(intensity_cartesian)
        ax[0].set_xlim(-smax, smax)
        ax[0].set_ylim(-smax, smax)
        ax[0].set_title(f"{axis_name} = {axis_value}")
        ax[0].set_aspect('equal',adjustable='box')
        plt.show()
